* openai nécessite un plan de facturation pour utiliser une API key
* mistral nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois
* anthropic nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois
* google_generative_ai nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois

In [192]:
# choix du projet
projectDir = "project-random"

In [193]:
# choix du fournisseur d'ia
provider = "mistral" # openai, mistral, anthropic, google_generative_ai

In [194]:
# changement du répertoire de travail (pour l'execution des outils et la lecture des fichiers)
import os
import sys

root = os.path.abspath(os.path.dirname(__vsc_ipynb_file__))
projectPath = os.path.abspath(f"{root}/{projectDir}")
projectFile = os.path.abspath(f"{projectPath}/project.md")
print("Répertoire racine:", root)
print("Projet:", projectDir)
print("Chemin projet:", projectPath)
print("Fichier projet:", projectFile)
print("Répertoire de travail:", projectDir)

# pour un accès direct aux fichiers du projet
os.chdir(projectPath)

# pour importlib.import_module puisse trouver les modules d'outils
if root not in sys.path:
    sys.path.append(root)

Répertoire racine: c:\Users\aceteam\source\repos\ai-workflow\test 2 - middleware
Projet: project-random
Chemin projet: c:\Users\aceteam\source\repos\ai-workflow\test 2 - middleware\project-random
Fichier projet: c:\Users\aceteam\source\repos\ai-workflow\test 2 - middleware\project-random\project.md
Répertoire de travail: project-random


In [195]:
# affiche les versions installées de python, langchain et l'executable utilisé

import sys
print(sys.version)

import langchain
print(langchain.__version__)

import sys
print(sys.executable)

3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
1.2.13
c:\Users\aceteam\anaconda3\envs\ai-workflow-test2\python.exe


In [196]:
# vérifie la présence des API key
import os

# autres intégrations
# https://docs.langchain.com/oss/python/integrations/chat/anthropic
# https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
# https://docs.langchain.com/oss/python/integrations/chat/mistralai


# Assure-toi d'avoir ton API key
if provider == "openai" and not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Please set your OpenAI API key in the environment variable OPENAI_API_KEY")
if provider == "mistral" and not os.environ["MISTRAL_API_KEY"]:
    raise ValueError("Please set your OpenAI API key in the environment variable MISTRAL_API_KEY")

In [197]:
# fonction: parse une chaine
import re
def parse_string(template, variables):
    # Regex pour trouver les zones de code entre backticks
    pattern = r"`([^`]+)`"

    def replace_code(match):
        code = match.group(1)  # extrait le contenu sans les backticks
        try:
            # évalue le template avec str.format et les variables
            return code.format(**variables)
        except KeyError as e:
            return code  # si variable non définie, on garde le code tel quel
    
    return re.sub(pattern, replace_code, template)

In [198]:
# fonctions: recherche dans les titres et lignes
import re

# obtient un élément de texte
def extract_text(lines, start_line, end_line):
    return "\n".join(lines[start_line:end_line+1]).strip()

# obtient un élément de code entre ``` et ```
def extract_code(lines, start_line, end_line):
    code_lines = []
    in_code_block = False
    for line in lines[start_line:end_line+1]:
        if line.strip().startswith("```"):
            in_code_block = not in_code_block
            continue
        if in_code_block:
            code_lines.append(line)
    return "\n".join(code_lines).strip()

# extrait un tableau markdown en une liste de dictionnaires
def extract_table(lines, start_line, end_line):
    table=[]
    headers = None
    # parcours des lignes du bloc
    for i in range(start_line, end_line):
        line = lines[i].strip()

        # ignorer les lignes vides
        if not line:
            continue

        # détecte les lignes du tableau
        if '|' in line:
            # découpe les colonnes et supprime les espaces
            row = [c.strip() for c in line.strip('|').split('|')]

            # première ligne du tableau = headers
            if headers is None:
                headers = row
                continue

            # ignorer la ligne de séparateurs --- | --- | ---
            if all(re.fullmatch(r'-+', c) for c in row):
                continue

            # remplir le jalon avec les colonnes présentes dans create_jalon
            task = {}
            for h, v in zip(headers, row):
                key = h.strip()
                task[key] = v

            table.append(task)

    return table

# trouve les lignes de début et de fin d'une section à partir de son titre
def find_section(titles, lines, section_title):
    si, st = next(((i,t) for i,t in enumerate(titles) if t['title'].lower() == section_title.lower()), (None, None))
    if st:
        ei, et = next(((i,t) for i,t in enumerate(titles) if i > si and t['level'] <= st['level']), (None, None))
        
        if not et:
            return (int(st['line'])+1, len(lines)-1)
        
        return (int(st['line'])+1, int(et['line'])-1)
    return (None, None)

In [199]:
# fonction: extrait les données d'un jalon dans les variables globales
def read_jalon(lines, start_line, end_line, numero, data):
    print(f"Lecture du jalon {numero} (lignes {start_line} à {end_line})")
    """
    Lit un bloc Jalon et extrait les informations du tableau Markdown.
    Remplit un dictionnaire de jalons et l'ajoute à workflow_data['jalons']
    """

    jalon = []

    headers = None

    # parcours des lignes du bloc
    for i in range(start_line, end_line):
        line = lines[i].strip()

        # ignorer les lignes vides
        if not line:
            continue

        # détecte les lignes du tableau
        if '|' in line:
            # découpe les colonnes et supprime les espaces
            row = [c.strip() for c in line.strip('|').split('|')]

            # première ligne du tableau = headers
            if headers is None:
                headers = row
                continue

            # ignorer la ligne de séparateurs --- | --- | ---
            if all(re.fullmatch(r'-+', c) for c in row):
                continue

            # remplir le jalon avec les colonnes présentes dans create_jalon
            task = {"Numéro": numero, "Job": "", "Acteur": "", "Skills": "", "Outils": "", "Contexte": "", "Etape": "", "Produit": "", "Itération": "1", "Validation": ""}
            for h, v in zip(headers, row):
                key = h.strip()
                if key in task:
                    task[key] = v

            jalon.append(task)

    data["jalons"].append(jalon)

In [200]:
# fonction: lit la totalité du document est extrait les informations dans les varaibles globales
import re

def read_workflow(titles, lines, data):
    """
    Cherche les sous-titres Jalon X dans les titres extraits et appelle read_jalon.
    """
    
    start_line, end_line = find_section(titles, lines, "Workflow")

    # pattern pour détecter Jalon 1, Jalon 2, ...
    jalon_pattern = re.compile(r'jalon\s*(\d+)', re.IGNORECASE)

    # filtrer les titres situés dans la section workflow
    workflow_titles = [t for t in titles if start_line <= t['line'] < end_line]

    # récupère les variables globales du workflow
    vars = {}
    vars_code = extract_code(lines, *find_section(titles, lines, "Variables"))
    if vars_code:
        exec(vars_code, {}, vars)
    data["variables"] = vars

    # récupère les fonctions globales du workflow
    funcs = {}
    display(extract_text(lines, *find_section(titles, lines, "Fonctions")))
    funcs_code = extract_code(lines, *find_section(titles, lines, "Fonctions"))
    if funcs_code:
        exec(funcs_code, funcs)
        funcs.pop("__builtins__", None)
    data["functions"] = funcs

    # ne garder que les titres qui sont des jalons
    jalons = []
    for t in workflow_titles:
        m = jalon_pattern.match(t['title'])
        if m:
            jalons.append({
                "line": t['line'],
                "numero": int(m.group(1))
            })

    # traiter chaque jalon
    for idx, j in enumerate(jalons):
        start = j["line"] + 1
        if idx + 1 < len(jalons):
            end = jalons[idx + 1]["line"]
        else:
            end = end_line

        read_jalon(lines, start, end, j["numero"], data)

In [ ]:
# fonction: teste la condition de validation d'un job
def check_job_validation(job, iteration, funcs, vars):
    allowed_globals = {
        "fileExists": lambda filename : os.path.isfile(filename),
        "build": lambda filename : os.path.isfile(filename),
        #,"multiply": multiply
    }

    allowed_globals.update(funcs)

    local_vars = vars | {"i": iteration}
    
    if job["Validation"] is not None:
        content = job["Validation"].strip()
        if content.startswith("`"):
            code = content.strip("`").strip()
            result = eval(code, allowed_globals, local_vars)
        else: 
            result =  False
            
        print("Validation content:", content, result, "with", local_vars)

        return result
    else:
        print(f"Job {job['Job']} du jalon {job['Jalon']} n'a pas de validation.")
        return True

In [202]:
# fonction: crée la base de données vectorielle avec les documents déjà produits
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader, UnstructuredMarkdownLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import glob
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import os

database_filename = "vector_db"

# modèle LLM utilisé
# les embeddings sont spécifiques au modèle utilisé pour les créer
llm = None

if provider == "mistral":
    llm = ChatMistralAI(model="mistral-large-latest")
if provider == "openai":
    llm = ChatOpenAI(model="gpt-4o-mini")

# Créer les embeddings
# Les embeddings permettent de comparer la similarité sémantique
# les embeddings sont spécifiques au modèle utilisé pour les créer
embeddings = None

if provider == "mistral":
    embeddings = MistralAIEmbeddings()
if provider == "openai":
    embeddings = OpenAIEmbeddings()

if not llm or not embeddings:
    raise ValueError("implémentation manquante pour le modèle choisi: " + provider)

# crée la base de données contextuelle
def create_database():
    docs = []

    ignores = ["journal.md", "project.md"]

    # charge les documents du contexte
    # Chaque fichier devient un Document LangChain
    for file in glob.glob("*.txt"):
        if not file in ignores:
            loader = TextLoader(file)
            docs.extend(loader.load())
            #docs[-1].metadata["acteur"] = job["Acteur"]
        
    for file in glob.glob("*.md"):
        if not file in ignores:
            loader = UnstructuredMarkdownLoader(file, mode="elements")
            docs.extend(loader.load())
            #docs[-1].metadata["acteur"] = job["Acteur"]

    for file in glob.glob("*.csv"):
        if not file in ignores:
            loader = CSVLoader(file, mode="elements")
            docs.extend(loader.load())
            #docs[-1].metadata["acteur"] = job["Acteur"]

    for doc in docs:
        doc.metadata["name"] = os.path.basename(doc.metadata["source"])


    # retourne une base de données vide
    if not docs:
        dummy = [Document(page_content="init")]
        db = FAISS.from_documents(dummy, embeddings)
        db.delete([list(db.docstore._dict.keys())[0]])
        return db

    # découpe les documents en chunks pour les fournir à l'agent
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    #tool_docs = text_splitter.create_documents([d.page_content for d in docs])
    chunks = text_splitter.split_documents(docs)

    # Crée la base vectorielle
    # la base vectoriel est spécifique au modèle utilisé pour les embeddings
    db = FAISS.from_documents(chunks, embeddings)
    
    # Sauvegarde la base vectorielle
    db.save_local(database_filename)

    return db

# charge la base de données contextuelle
def load_database():
    # Sauvegarde la base vectorielle
    db = FAISS.load_local(
        database_filename,
        embeddings,
        allow_dangerous_deserialization=True
        )
    return db

La fonction `execute_job` inbtéroge le LLM en passant à l'agent les outils et le contexte donné en argument. 

La réponse et une discution, les éléments produits sont généralement des fichiers nommé explicitement par le Job

In [203]:
# fonction: execute un job

from langchain.agents import create_agent
from datetime import datetime
import importlib
import os

# evite les injection de path pour les outils
def is_safe_tool_name(name):
    return re.fullmatch(r"[a-zA-Z0-9_\-]+", name) is not None

# exécute un job
def execute_job(titles, lines, job, iteration, vars):

    local_vars = vars | {"i": iteration}

    Etape = parse_string(job["Etape"], local_vars)
    Produit = parse_string(job["Produit"], local_vars)
    Acteur = parse_string(job['Acteur'], local_vars)
    Contexte = parse_string(job['Contexte'], local_vars)
    Outils = parse_string(job['Outils'], local_vars)

    tools = []
    
    print("outils:", re.split(r"[;,|]", Outils))
    for tool in re.split(r"[;,|]", Outils):
        tool = tool.strip()

        if( not tool):
            continue

        if not is_safe_tool_name(tool):
            raise ValueError("not safe tool name:", tool)
        
        filename = root+"/tools/"+tool+".py"
        if not os.path.isfile(filename):
            raise ValueError("tool file not found:", filename)
        
        modulename = "tools."+tool

        # importer le module (cache initial)
        module = importlib.import_module(modulename)
                
        # recharger le module pour forcer la mise à jour
        module = importlib.reload(module)

        value = getattr(module, "tools", None)
        if value and isinstance(value, list):
            tools.extend(value)
        else:
            raise ValueError("no tools list found in module:", modulename)

        print("add tool:", tool)

    print("tools:", tools)

    db = create_database()
    
    # Créer le retriever
    # (récupérer k passages pertinents par contexte)
    docs = []
    contexts = []
    files = []

    for src in re.split(r"[;,|]", Contexte):
        src = src.strip()
        
        if( not src):
            continue

        if not os.path.exists(src):
            # extraire le contexte du workflow
            start, end = find_section(titles, lines, src)
            if not start or not end:
                raise ValueError("context section not found: " + src)
            context = extract_text(lines, start, end)
            contexts.append(context)
        else:
            files.append(src)
            print("search in db for context:", src)
            retriever = db.as_retriever(
                search_kwargs={"k": 2, "filter": {"filename": src}}
            )
            docs.extend(retriever.invoke(job["Job"]))

    print("contexts:", contexts)
    print("docs:", docs)

    # Trie les résultats par score
    docs = sorted(docs, key=lambda d: d.metadata.get("score", 0), reverse=True)

    # extrait les passages pertinants pour le contexte du prompt
    prompt_context_docs = ""
    for d in docs:
        print(d)
        prompt_context += f"## {d.metadata['name']}:\n\n```\n{d.page_content}\n```\n\n"
    
    # extrait les passages pertinants pour le contexte du prompt
    prompt_context_general = "\n".join(contexts)
    
    # extrait les passages pertinants pour le contexte du prompt
    prompt_context_files = ", ".join(files)
    
    # Créer un agent
    # permet d'utiliser les aller-retours avec des outils et skills
    agent = create_agent(llm, tools=tools)

    prompt_inputs = {
        "messages": [
            {"role": "system", "content": f"""
Tu es {Acteur} et tu dois réaliser ce travail: {Etape}.

Utilise les outils mise à ta disposition pour accomplir cette tâche. Les outils disponibles sont: {Outils}.

Les fichiers suivants peuvent t'aider à contextualiser cette tâche: {prompt_context_files}.

# Contexte général
{prompt_context_general}

# Documents existants
{prompt_context_docs}
"""},
            {"role": "user", "content": f"Ce travail doit produire exactement ce résultat: {Produit}"}
        ]
    }

    print("prompt_inputs:", prompt_inputs)

    response = agent.invoke(prompt_inputs)

    print("response:", response)

    with open("journal.md", "a", encoding="utf-8") as f:
        f.write(f"\n--- {datetime.now()} ---\n")
        f.write(f"job: {job['Job']}\n")
        f.write("\n\n")

        for msg in response["messages"]:
            f.write(msg.__class__.__name__ + ":\n")
            f.write(msg.content + "\n")

    return True 

In [204]:
# extrait les titres et leur niveau de lal iste de lignes
def extract_titles(lines):
    """
    Extrait les titres Markdown et leur niveau en ignorant les titres
    à l'intérieur des blocs de code (```) dans une liste de lignes.
    """
    titles = []
    in_code_block = False

    for num_ligne, line in enumerate(lines):
        stripped_line = line.strip()

        # Détecter le début/fin d'un bloc de code
        if stripped_line.startswith("```"):
            in_code_block = not in_code_block
            continue  # On ne traite pas cette ligne comme un titre

        # Ignorer les lignes dans un bloc de code
        if in_code_block:
            continue

        # Détecter un titre Markdown
        if stripped_line.startswith('#'):
            # Compter le nombre de # pour déterminer le niveau
            level = 0
            while level < len(stripped_line) and stripped_line[level] == '#':
                level += 1

            # Extraire le texte du titre
            title_text = stripped_line[level:].strip()
            titles.append({
                'line': num_ligne,
                'level': level,
                'title': title_text
            })

    return titles

In [205]:
lines = []
with open(projectFile, "r", encoding="utf-8") as fichier:
    contenu = fichier.read()
    lines = contenu.splitlines() # Supprime tous les types de sauts de ligne (\n, \r, \r\n)

titles = extract_titles(lines)

display(titles)

workflow_data = {
    "jalons": [],
    "variables": {},
    "functions": {}
}

read_workflow(titles, lines, workflow_data)

display(workflow_data)

# execute les jobs un à un 
for jalon in workflow_data["jalons"]:
    for job in jalon:
        i = 0
        j = int(parse_string(job["Itération"], workflow_data["variables"] | {"i": i}))

        for i in range(j):
            if not check_job_validation(job, i, workflow_data["functions"], workflow_data["variables"]):
                print(f"{job['Job']} => Execution...")
                result = execute_job(titles, lines, job, i, workflow_data["variables"])
                if result:
                    print(f"=> Validé")

                    reponse = input("Voulez-vous continuer ? (o/n) : ")
                    if reponse.lower() in ["o", "oui", "y", "yes"]:
                        print("On continue !")
                    else:
                        raise ValueError("Opération annulée.")
                else:
                    raise ValueError(f"=> Échec de l'exécution. {result}")
            else:
                print(f"{job['Job']} => OK")

[{'line': 0, 'level': 1, 'title': 'Processus exécution de systèmes découplés'},
 {'line': 2, 'level': 2, 'title': 'Contexte'},
 {'line': 4, 'level': 3, 'title': 'Sujet'},
 {'line': 8, 'level': 3, 'title': 'Le jeu'},
 {'line': 14, 'level': 3, 'title': 'Résultat attendu'},
 {'line': 18, 'level': 3, 'title': 'Les parties prenantes (acteurs):'},
 {'line': 24, 'level': 2, 'title': 'Jalons'},
 {'line': 34, 'level': 2, 'title': 'Workflow'},
 {'line': 36, 'level': 3, 'title': 'Variables'},
 {'line': 73, 'level': 3, 'title': 'Fonctions'},
 {'line': 135, 'level': 3, 'title': 'Jalon  1'},
 {'line': 141, 'level': 3, 'title': 'Jalon  2'},
 {'line': 149, 'level': 2, 'title': 'Skills'},
 {'line': 153, 'level': 2, 'title': 'Tools'},
 {'line': 157, 'level': 1, 'title': 'Mise en place des agents IA'}]

'Fonctions utilisées pour la validation des étapes.\n\n```python\n#\n# Recherche les lignes dans un fichier texte\n#\n\ndef contains(fichier, lignes_attendues):\n    """\n    Vérifie que chaque ligne de \'lignes_attendues\' est présente dans \'fichier\'.\n    Ignore la casse et les espaces en début/fin de ligne.\n    \n    :param fichier: chemin vers le fichier à vérifier\n    :param lignes_attendues: liste de chaînes de caractères à vérifier\n    :return: True si toutes les lignes sont présentes, False sinon\n    """\n    try:\n        with open(fichier, "r", encoding="utf-8") as f:\n            contenu = f.readlines()\n    except FileNotFoundError:\n        print(f"Erreur : le fichier {fichier} n\'existe pas.")\n        return False\n\n    # Nettoyer le contenu : trim et minuscules\n    contenu_nettoye = [line.strip().lower() for line in contenu]\n\n    # Vérifier chaque ligne attendue\n    for ligne in lignes_attendues:\n        if ligne.strip().lower() not in contenu_nettoye:\n    

Lecture du jalon 1 (lignes 136 à 141)
Lecture du jalon 2 (lignes 142 à 148)


{'jalons': [[{'Numéro': 1,
    'Job': 'trouver les pièces',
    'Acteur': 'Joueur',
    'Skills': '',
    'Outils': 'write_file, read_file',
    'Contexte': 'Sujet, Le jeu, article1.txt, article2.txt, article3.txt',
    'Etape': 'Rechercher dans les fichiers disponibles le nom des pièces du puzzle.',
    'Produit': '1 fichier `pieces.txt` avec la liste des pièces (1 par ligne).',
    'Itération': '1',
    'Validation': '`contains("pieces.txt", ["chien", "souris", "puce", "éléphant", "tortue"])`'}],
  [{'Numéro': 2,
    'Job': 'implémentation prototypes',
    'Acteur': 'Joueur',
    'Skills': '',
    'Outils': 'write_file, read_file',
    'Contexte': 'Sujet, Le jeu, pieces.txt',
    'Etape': "Trouver l'ordre des pièces du puzzle",
    'Produit': "1 fichier `ordre.txt` avec la liste des pièces (1 par ligne) dans l'ordre souhaité",
    'Itération': '1',
    'Validation': '`containsInOrder("pieces.txt", ["puce", "souris", "chien", "tortue", "éléphant"])`'}]],
 'variables': {'niv': 4, 'anim

'content'

...

Validation content: `contains("pieces.txt", ["chien", "souris", "puce", "éléphant", "tortue"])` True with {'niv': 4, 'anims': 8, 'pieges': 40, 'platformes': 20, 'i': 0}
trouver les pièces => OK


'content'

'`containsInOrder("pieces.txt", ["puce", "souris", "chien", "tortue", "éléphant"])`'

Validation content: `containsInOrder("pieces.txt", ["puce", "souris", "chien", "tortue", "éléphant"])` False with {'niv': 4, 'anims': 8, 'pieges': 40, 'platformes': 20, 'i': 0}
implémentation prototypes => Execution...
outils: ['write_file', ' read_file']
add tool: write_file
add tool: read_file
tools: [StructuredTool(name='write_file', description='Ecrit un texte dans un fichier\n\n    Args:\n        filename: Nom du fichier dans lequel écrire le texte\n        text: Texte à écrire dans le fichier', args_schema=<class 'langchain_core.utils.pydantic.write_file'>, func=<function write at 0x000001DF06649260>), StructuredTool(name='read_file', description='Lit un fichier texte\n\n    Args:\n        filename: Nom du fichier à lire', args_schema=<class 'langchain_core.utils.pydantic.read_file'>, func=<function read at 0x000001DF06361DA0>)]
search in db for context: pieces.txt
contexts: ['Résoudre un puzzle.', "Tu dispose de fichiers contenant des articles de revues spécialisé. Dans ces text

ValueError: Opération annulée.